# 04 Embeddings, Eventhouse load, and vector search

| | |
|---|---|
| **Layer** | Gold-style serving (search index) |
| **Reads** | `silver_doc_chunks` |
| **Writes** | Eventhouse tables `DocChunksStage`, `DocChunksVec` |
| **Re-runnable (idempotent)** | Yes. The stage table is replaced into `DocChunksVec`. |

Each chunk is turned into a list of numbers (an embedding) that captures its meaning. Questions are embedded with the same model, and chunks are ranked by cosine similarity. Text with similar meaning lands close together, so "high blood sugar emergency" can find chunks about hyperglycemia without sharing those words.

**Before running:** create the Eventhouse tables with `eventhouse/setup.kql` (stage table first). After the load cell, run the conversion command from that file.

**Data policy:** use public sample data only. Do not send company documents to an external API without approval. A company deployment would normally use its approved, in-boundary embedding service (for example Azure OpenAI).

**Standards followed:** parameters at the top, no secrets in code, small documented functions, explicit schemas, a validation cell at the end, and a printed run summary.

## Parameters

In [ ]:
PROVIDER        = "openai"                       # "openai" or "azure_openai"
EMBED_MODEL     = "text-embedding-3-small"       # openai model name, or the Azure deployment name
AZURE_ENDPOINT  = ""                             # azure_openai only, e.g. https://<resource>.openai.azure.com
AZURE_API_VER   = "2024-02-01"                   # azure_openai only; confirm the current version
CHUNKS_TABLE    = "silver_doc_chunks"
KUSTO_URI       = "<your Eventhouse query URI>"  # https://...kusto.fabric.microsoft.com (no trailing slash, no "ingest-")
KUSTO_DB        = "<your KQL database name>"
STAGE_TABLE     = "DocChunksStage"
BATCH_SIZE      = 32

## API key
Typed at run time and never saved. In Fabric, if the prompt does not appear, paste the key into the variable, run the cell, then delete the pasted text before saving or sharing. With Key Vault you would fetch it by secret name instead.

In [ ]:
import getpass
API_KEY = getpass.getpass("Embedding API key: ")

## Read chunks

In [ ]:
import json, time, requests

chunks = spark.table(CHUNKS_TABLE).toPandas()       # small table, safe to bring to the driver
texts = chunks["chunk_text"].tolist()
print(len(texts), "chunks")

## Embedding helper
Retries rate limits (429) and server errors (5xx) with backoff. Other errors fail fast, because retrying a bad key or bad request never helps.

In [ ]:
def embed(batch, tries: int = 5):
    """Return one embedding per input text, in the same order."""
    if PROVIDER == "openai":
        url, headers = "https://api.openai.com/v1/embeddings", {"Authorization": f"Bearer {API_KEY}"}
        body = {"model": EMBED_MODEL, "input": batch}
    elif PROVIDER == "azure_openai":
        url = f"{AZURE_ENDPOINT}/openai/deployments/{EMBED_MODEL}/embeddings?api-version={AZURE_API_VER}"
        headers, body = {"api-key": API_KEY}, {"input": batch}
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

    for attempt in range(1, tries + 1):
        resp = requests.post(url, headers=headers, json=body, timeout=60)
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < tries:
            time.sleep(2 ** attempt)
            continue
        resp.raise_for_status()                                  # 401 = bad key; 429 after retries = quota/billing
        return [d["embedding"] for d in sorted(resp.json()["data"], key=lambda d: d["index"])]

In [ ]:
vectors = []
for i in range(0, len(texts), BATCH_SIZE):
    vectors.extend(embed(texts[i:i + BATCH_SIZE]))
    print(f"embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)}")

assert len(vectors) == len(texts), "one vector per chunk expected"
print("dimensions:", len(vectors[0]))

## Write to the Eventhouse stage table
Vectors are written as JSON text and converted to a `dynamic` column by the KQL command in `eventhouse/setup.kql`. This avoids type-mapping surprises in the connector.

In [ ]:
out = chunks.copy()
out["content_vector_json"] = [json.dumps(v) for v in vectors]
stage_df = spark.createDataFrame(out[["chunk_id", "doc_id", "file_name", "page_number", "chunk_index",
                                      "chunk_text", "word_count", "batch_id", "content_vector_json"]])

token = notebookutils.credentials.getToken(KUSTO_URI)
(stage_df.write.format("com.microsoft.kusto.spark.synapse.datasource")
    .option("kustoCluster", KUSTO_URI)
    .option("kustoDatabase", KUSTO_DB)
    .option("kustoTable", STAGE_TABLE)
    .option("accessToken", token)
    .option("tableCreateOptions", "FailIfNotExist")     # the table must be created on purpose
    .mode("Append")
    .save())
print("written", stage_df.count(), "rows to", STAGE_TABLE)

Now run the `.set-or-replace DocChunksVec` command and the check from `eventhouse/setup.kql`. The check should show 1536 dimensions for `text-embedding-3-small`.

## Search by meaning
OpenAI embeddings are normalized, so the norm arguments to `series_cosine_similarity` are 1.0. Scores run from -1 to 1. Good matches with real text are often 0.4 to 0.7, so judge by ranking and gaps, not by closeness to 1.

In [ ]:
def search(question: str, top: int = 5):
    """Embed the question and return the top chunks by cosine similarity."""
    query_vector = embed([question])[0]
    kql = f"""
    let qv = dynamic({json.dumps(query_vector)});
    DocChunksVec
    | extend sim = series_cosine_similarity(qv, content_vector, 1.0, 1.0)
    | top {top} by sim desc
    | project sim, file_name, page_number, chunk_text
    """
    token = notebookutils.credentials.getToken(KUSTO_URI)
    return (spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
            .option("kustoCluster", KUSTO_URI).option("kustoDatabase", KUSTO_DB)
            .option("accessToken", token).option("kustoQuery", kql).load())


display(search("high blood sugar emergency"))

## Retrieval checks
A small set of questions with a word that should appear in the top results. A MISS is information, not failure: read the top chunks to see whether the search or the test word was wrong. Note that sources may use abbreviations (for example `T1` for type 1), so tests need to know them.

In [ ]:
tests = [
    # (question, text expected in the top 3 chunks)
    ("high blood sugar emergency",              "hyperglycemia"),
    ("diabetic ketoacidosis in children",       "ketoacidosis"),
    ("non-ketotic hyperglycemia",               "non-ketotic"),
    ("brain imaging in a diabetic patient",     "MRI"),
    ("involuntary movements in diabetes",       "movement"),
    ("type 1 diabetes complications",           "T1"),
    ("brain damage from very high glucose",     "cortical"),
]
passed = 0
for question, expected in tests:
    top = search(question, top=3).toPandas()
    hit = top["chunk_text"].str.contains(expected, case=False).any() or top["file_name"].str.contains(expected, case=False).any()
    passed += hit
    print("PASS" if hit else "MISS", "|", question)
print(f"{passed}/{len(tests)} passed")